# Regex in Python: finding patterns in statement text

Last session you learned the core loop: compile a pattern, match it against a string, and pull the result out of the match object. This notebook consolidates that and builds the toolkit you'll lean on once we start pulling fields out of messy transaction text.

Companion reading is the official Python **Regular Expression HOWTO** (docs.python.org/3/howto/regex.html), the same reference you used. This notebook follows its arc: simple patterns first, then how to use them in Python, then a little more power.

**Where regex fits in this course.** It is the tool for *unstructured* text, like transaction lines copied out of a PDF. Once the bank hands us a clean CSV, most of this work disappears, and that contrast is the whole point of the next lesson. Reach for regex when the data is messy and nobody split it into columns for you.

Run the cells top to bottom.

## Raw strings: always write patterns as `r"..."`

Regex leans on backslashes (`\d`, `\s`, `\b`). Python *also* uses backslashes for its own escapes (`\n`, `\t`). When you write a pattern in an ordinary string the two collide, which the HOWTO calls "the backslash plague."

The fix is the `r` prefix, a **raw string**, which tells Python to leave backslashes alone and pass them straight to the regex engine. Make it a habit: every pattern starts with `r`.

In [ ]:
# Python's own escapes clash with regex escapes.
# \t is a tab to Python, nothing to do with regex:
print(repr("a\tb"))     # the \t became a real tab character
print(repr(r"a\tb"))    # raw string keeps backslash-t literally, which is what a pattern wants

# \b is the worst offender: a backspace to Python, a word boundary to regex
print(repr("\b"), "length", len("\b"))     # control character, length 1
print(repr(r"\b"), "length", len(r"\b"))   # backslash + b, length 2 -- what the engine expects

## The core loop you already know: compile -> match -> group

`re.compile` turns a pattern string into a reusable pattern object. `.match` tries the pattern at the **start** of a string and returns a *match object* (or `None`). `.group()` pulls the matched text out of that object.

Compiling once and reusing is the normal style when you'll apply the same pattern to many lines of a statement.

In [ ]:
import re

pattern = re.compile(r"\d{2}/\d{2}/\d{4}")   # two digits / two digits / four digits

m = pattern.match("06/05/2026 PAYROLL NORTHWIND TRADING CO")
print("matched?", m is not None)
print("text   :", m.group())     # the matched substring
print("span   :", m.span())      # (start, end) indices
print("start  :", m.start(), " end:", m.end())

# No match returns None. Always check before calling .group(), or you hit AttributeError.
print(pattern.match("PAYROLL on 06/05/2026"))   # date isn't at the start, so .match fails

## Metacharacters: the special cast

Most characters match themselves. A handful are **special**:

`. ^ $ * + ? { } [ ] \ | ( )`

They mean things like "any character," "start," "one or more," and "a set of characters." The next cells walk the ones you'll actually use on statement data. To match a metacharacter literally, put a backslash in front: `\.` is a real dot, `\$` a real dollar sign, `\(` a real parenthesis, which you'll need for those `($64.76)` debits.

## Character classes `[...]` and shorthands

Square brackets define a **set** to match one of. `[abc]` matches a single `a`, `b`, or `c`. Ranges work: `[0-9]`, `[a-z]`, `[A-Za-z]`. A leading `^` negates the set, so `[^0-9]` matches anything that is not a digit.

Common sets have shorthands, straight from the HOWTO:
- `\d` a digit, same as `[0-9]`
- `\w` a word character (letters, digits, underscore)
- `\s` whitespace (space, tab, newline)

Their uppercase versions negate: `\D` non-digit, `\W` non-word, `\S` non-space.

In [ ]:
import re

# \d matches one digit; \D matches one non-digit
print(re.compile(r"\d").findall("88X2"))    # ['8', '8', '2']
print(re.compile(r"\D").findall("88X2"))    # ['X']

# a custom set: just the vowels
print(re.compile(r"[aeiou]").findall("transaction"))    # ['a', 'a', 'i', 'o']

# a negated set: characters that are NOT digits or spaces
print(re.compile(r"[^\d ]").findall("06/05/2026"))      # ['/', '/']

## Quantifiers: how many times

Quantifiers say how many of the preceding thing to match:
- `*` zero or more
- `+` one or more
- `?` zero or one (optional)
- `{n}` exactly n; `{m,n}` between m and n

So `\d{4}` is exactly four digits (a year), `\d+` is one or more digits, and `\(?` is an optional opening parenthesis, handy because debits look like `($64.76)` while credits have no parens.

**Greedy vs lazy.** By default quantifiers are *greedy*: they grab as much as possible, then back up if the rest of the pattern fails (the HOWTO walks through exactly this backtracking). Add `?` after a quantifier to make it *lazy*, grabbing as little as possible. The difference bites constantly in real parsing.

In [ ]:
import re

text = "<first> <second>"
print(re.compile(r"<.*>").findall(text))    # greedy: ['<first> <second>'] -- grabbed everything
print(re.compile(r"<.*?>").findall(text))   # lazy:   ['<first>', '<second>']

# an optional paren in front, a step toward matching a debit amount
print(re.compile(r"\(?\$\d").findall("paid ($5 and $9"))   # ['($5', '$9']

## Anchors: position without consuming characters

- `^` matches at the start of the string
- `$` matches at the end
- `\b` matches a **word boundary**, the edge between a word character and a non-word character

Anchors match *positions*, not characters. `\b` is great for whole-token matches: `\bACH\b` finds the standalone token `ACH` but not the `ACH` buried inside a longer word.

In [ ]:
import re

print(re.compile(r"^06").match("06/05/2026") is not None)      # True: starts with 06
print(re.compile(r"2026$").search("06/05/2026") is not None)   # True: ends with 2026

# word boundaries: match ACH as its own token, not inside ACHIEVE
print(re.compile(r"\bACH\b").findall("PAYROLL ACH CREDIT and ACHIEVE"))   # ['ACH']

## `match` vs `search` vs `fullmatch` (a classic trap)

The HOWTO flags this one because it catches almost everyone:
- `.match` only looks at the **start** of the string
- `.search` scans the **whole** string for the first match
- `.fullmatch` requires the pattern to cover the **entire** string

You used `.match` last time, which is perfect when the thing you want sits at the front of the line. When a field is buried mid-string, `.match` returns `None` and `.search` is what you want. When we recover a missing merchant name next lesson, if the name sits at the *front* of the description, then `.match` is the right call there.

In [ ]:
import re

p = re.compile(r"\d{2}/\d{2}/\d{4}")
line = "PAYROLL posted on 06/05/2026"

print("match :", p.match(line))     # None -- the date is not at the start
print("search:", p.search(line))    # finds it mid-string
print("found :", p.search(line).group())

## Finding every match, not just the first

A statement has dozens of dates and amounts. `.match` and `.search` return *one* result; you usually want all of them.
- `.findall` returns a list of the matched strings
- `.finditer` returns match objects one at a time, so you also get positions

Reach for `.finditer` when you want to loop over every hit and know *where* each one is.

In [ ]:
import re

# a few lines as you might copy them out of a PDF statement
statement = '''06/01/2026  BLUE HERON PROPERTY MGMT RENT 06-01 ACH DEBIT      ($1,650.00)
06/05/2026  PAYROLL NORTHWIND TRADING CO 88X2 TRAN ALEX MORGAN     $1,875.40
06/05/2026  CLEARWAVE MOBILE 06-05 800-555-0142 GA DEBIT CARD        ($72.50)
06/12/2026  ATM WITHDRAWAL 06-12 1ST AVE ATLANTA GA                 ($100.00)'''

dates   = re.compile(r"\d{2}/\d{2}/\d{4}")
amounts = re.compile(r"\(?\$[\d,]+\.\d{2}\)?")

print("all dates  :", dates.findall(statement))
print("all amounts:", amounts.findall(statement))
print()
for m in amounts.finditer(statement):
    print(f"{m.group():>12}  at characters {m.span()}")

## Two extras you'll want soon

**Flags** change matching behavior. The most common is `re.IGNORECASE`, so `ach` matches `ACH`. Pass it to `re.compile`.

**Substitution** with `.sub` replaces matches, the start of *cleaning* text rather than just reading it. Heads up though: for simple fixed-format cleanup, like stripping `$` and commas out of an amount, plain string methods (`.replace`) are usually clearer than a regex. Knowing *when not* to reach for regex is part of using it well, and you'll see that judgment call again in the CSV lesson. (The HOWTO makes the same point in its "Use string methods" section.)

In [ ]:
import re

print(re.compile(r"ach", re.IGNORECASE).findall("ACH credit, ach debit"))   # ['ACH', 'ach']

# stripping $ and commas with sub -- works, though .replace would be just as clear here
cleaned = re.sub(r"[$,]", "", "$1,875.40")
print(cleaned, "->", float(cleaned))    # 1875.40 -> 1875.4

## A repeatable workflow for writing patterns

This is the method worth teaching her, and using yourself:

1. **Look at real examples.** Copy a handful of actual lines from the statement and lay them side by side.
2. **Write the capture rules on paper, in plain words.** "A date is two digits, slash, two digits, slash, four digits." Describe each field before touching syntax.
3. **Translate one rule at a time** into a pattern, simplest first.
4. **Validate interactively.** Paste the pattern and sample text into pythex.org and watch what highlights. Adjust until only the right text lights up.
5. **Bring it into Python,** compile it, and run it over more lines to catch the cases your samples missed.

Refining a pattern a reference or an assistant suggests is fine, but draft it yourself first. Generating one from scratch skips the part where the syntax actually sticks.

## Where this points next

You can now *find* any pattern in statement text: a date, an amount, a phone number, a token like `ACH`. The next step is to *pull the pieces out* of what you matched, not just "this line contains a merchant name" but "the merchant name is `CLEARWAVE MOBILE`."

That is done with **capture groups**: parentheses around the part you want, and `.group(1)` to extract it. It is a small addition to everything here, and it is exactly how we recover missing merchant names in the CSV lesson. You are set up for it now.

## Practice

Use the `statement` text from the finditer cell (or paste new lines):

1. Find the **phone numbers** like `800-555-0142`.
2. Find the internal **reference codes** like `88X2` (digits, a letter, a digit).
3. Find every **transaction type word**, `DEBIT` or `CREDIT`, case-insensitively.
4. Trickier: match an amount **only when it is a debit** (wrapped in parentheses). What changes from the general amount pattern?

Draft each on paper, test on pythex, then confirm in the cell below.

In [ ]:
import re

statement = '''06/05/2026  CLEARWAVE MOBILE 06-05 800-555-0142 GA DEBIT CARD        ($72.50)
06/05/2026  PAYROLL NORTHWIND TRADING CO 88X2 TRAN ALEX MORGAN     $1,875.40'''

# 1. phone numbers (worked example to get you started):
print(re.compile(r"\d{3}-\d{3}-\d{4}").findall(statement))   # ['800-555-0142']

# 2. reference codes like 88X2  -> your turn:
# print(re.compile(r"...").findall(statement))

# 3. DEBIT or CREDIT, case-insensitive -> your turn:
# print(re.compile(r"...", re.IGNORECASE).findall(statement))

# 4. debit-only amounts, must have the parentheses -> your turn:
# print(re.compile(r"...").findall(statement))